In [1]:
def generate_number_animation(
    output_dir=".",
    mp4_name="numbers_categories_9x16.mp4",
    gif_name="numbers_categories_9x16.gif",
    preview_name="numbers_categories_preview.png",
    width_pixels=360,
    height_pixels=640,
    dpi=100,
    fps_mp4=10,
    fps_gif=8,
    reveal_frames=3,
    hold_frames=4,
    save_mp4=True,
    save_gif=True,
    save_preview=True,
    print_categories=False
):
    from pathlib import Path
    import math

    import matplotlib.pyplot as plt
    from matplotlib.animation import FuncAnimation, PillowWriter, FFMpegWriter
    from matplotlib.colors import to_rgba

    # =====================================================
    # FUNÇÕES AUXILIARES DE TEORIA DOS NÚMEROS
    # =====================================================
    def is_prime(n):
        if n < 2:
            return False

        if n % 2 == 0:
            return n == 2

        limit = math.isqrt(n)

        for divisor in range(3, limit + 1, 2):
            if n % divisor == 0:
                return False

        return True

    def prime_factors_with_multiplicity(n):
        factors = []
        value = n
        divisor = 2

        while divisor * divisor <= value:
            while value % divisor == 0:
                factors.append(divisor)
                value //= divisor

            divisor += 1 if divisor == 2 else 2

        if value > 1:
            factors.append(value)

        return factors

    def proper_divisor_sum(n):
        if n == 1:
            return 0

        total = 1
        limit = math.isqrt(n)

        for divisor in range(2, limit + 1):
            if n % divisor == 0:
                quotient = n // divisor
                total += divisor

                if quotient != divisor:
                    total += quotient

        return total

    def digit_sum(n):
        return sum(int(digit) for digit in str(n))

    def is_composite(n):
        return n > 1 and not is_prime(n)

    def is_semiprime(n):
        return (
            n >= 4
            and len(prime_factors_with_multiplicity(n)) == 2
        )

    def is_square(n):
        root = math.isqrt(n)
        return root * root == n

    def is_cube(n):
        approximate_root = round(n ** (1 / 3))

        for root in range(
            max(0, approximate_root - 1),
            approximate_root + 2
        ):
            if root**3 == n:
                return True

        return False

    def is_perfect_power(n):
        if n < 4:
            return False

        maximum_exponent = int(math.log2(n))

        for exponent in range(2, maximum_exponent + 1):
            approximate_base = round(n ** (1 / exponent))

            for base in range(
                max(2, approximate_base - 1),
                approximate_base + 2
            ):
                if base**exponent == n:
                    return True

        return False

    def triangular_numbers(limit):
        values = []
        index = 1

        while True:
            value = index * (index + 1) // 2

            if value > limit:
                break

            values.append(value)
            index += 1

        return values

    def tetrahedral_numbers(limit):
        values = []
        index = 1

        while True:
            value = index * (index + 1) * (index + 2) // 6

            if value > limit:
                break

            values.append(value)
            index += 1

        return values

    def square_pyramidal_numbers(limit):
        values = []
        index = 1

        while True:
            value = index * (index + 1) * (2 * index + 1) // 6

            if value > limit:
                break

            values.append(value)
            index += 1

        return values

    def pentagonal_numbers(limit):
        values = []
        index = 1

        while True:
            value = index * (3 * index - 1) // 2

            if value > limit:
                break

            values.append(value)
            index += 1

        return values

    def hexagonal_numbers(limit):
        values = []
        index = 1

        while True:
            value = index * (2 * index - 1)

            if value > limit:
                break

            values.append(value)
            index += 1

        return values

    def fibonacci_numbers(limit):
        values = []
        first = 1
        second = 1

        while first <= limit:
            values.append(first)
            first, second = second, first + second

        return sorted(set(values))

    def is_harshad(n):
        total = digit_sum(n)
        return total > 0 and n % total == 0

    def is_sphenic(n):
        factors = prime_factors_with_multiplicity(n)

        return (
            len(factors) == 3
            and len(set(factors)) == 3
        )

    def is_smith(n):
        if n < 4 or is_prime(n):
            return False

        factors = prime_factors_with_multiplicity(n)
        factor_digit_sum = sum(digit_sum(factor) for factor in factors)

        return digit_sum(n) == factor_digit_sum

    def is_binary_palindrome(n):
        binary = bin(n)[2:]
        return binary == binary[::-1]

    def is_decimal_palindrome(n):
        decimal = str(n)
        return decimal == decimal[::-1]

    def is_happy(n):
        visited = set()
        value = n

        while value != 1 and value not in visited:
            visited.add(value)

            value = sum(
                int(digit) ** 2
                for digit in str(value)
            )

        return value == 1

    def lucky_numbers(limit):
        values = list(range(1, limit + 1, 2))
        index = 1

        while index < len(values):
            step = values[index]

            if step > len(values):
                break

            values = [
                value
                for position, value in enumerate(values, start=1)
                if position % step != 0
            ]

            index += 1

        return set(values)

    def is_evil(n):
        return bin(n).count("1") % 2 == 0

    def is_automorphic(n):
        return str(n * n).endswith(str(n))

    def is_kaprekar(n):
        if n == 1:
            return True

        square = n * n
        number_digits = len(str(n))
        divisor = 10**number_digits

        left_part = square // divisor
        right_part = square % divisor

        return (
            right_part > 0
            and left_part + right_part == n
        )

    def catalan_numbers(limit):
        values = []
        catalan = 1
        index = 0

        while catalan <= limit:
            values.append(catalan)

            index += 1
            catalan = (
                catalan
                * 2
                * (2 * index - 1)
                // (index + 1)
            )

        return sorted(set(values))

    def pell_numbers(limit):
        values = []
        previous = 0
        current = 1

        while current <= limit:
            values.append(current)
            previous, current = current, 2 * current + previous

        return values

    # =====================================================
    # CONSTRUÇÃO DAS CATEGORIAS
    # =====================================================
    def build_categories(limit=100):
        numbers = list(range(1, limit + 1))
        lucky_set = lucky_numbers(limit)

        return [
            (
                "Natural Numbers",
                set(numbers),
                "#ff3b30"
            ),
            (
                "Odd Numbers",
                {n for n in numbers if n % 2 == 1},
                "#ff6b35"
            ),
            (
                "Even Numbers",
                {n for n in numbers if n % 2 == 0},
                "#ff9f0a"
            ),
            (
                "Multiples of 3",
                {n for n in numbers if n % 3 == 0},
                "#ffb300"
            ),
            (
                "Multiples of 5",
                {n for n in numbers if n % 5 == 0},
                "#ffcc00"
            ),
            (
                "Multiples of 7",
                {n for n in numbers if n % 7 == 0},
                "#ffd60a"
            ),
            (
                "Multiples of 10",
                {n for n in numbers if n % 10 == 0},
                "#fadb14"
            ),
            (
                "Prime Numbers",
                {n for n in numbers if is_prime(n)},
                "#d4ff00"
            ),
            (
                "Composite Numbers",
                {n for n in numbers if is_composite(n)},
                "#a3e635"
            ),
            (
                "Semiprime Numbers",
                {n for n in numbers if is_semiprime(n)},
                "#f4e285"
            ),
            (
                "Square Numbers",
                {n for n in numbers if is_square(n)},
                "#34c759"
            ),
            (
                "Cube Numbers",
                {n for n in numbers if is_cube(n)},
                "#30d158"
            ),
            (
                "Perfect Powers",
                {n for n in numbers if is_perfect_power(n)},
                "#00c853"
            ),
            (
                "Triangular Numbers",
                set(triangular_numbers(limit)),
                "#32d74b"
            ),
            (
                "Tetrahedral Numbers",
                set(tetrahedral_numbers(limit)),
                "#00b894"
            ),
            (
                "Square Pyramidal Numbers",
                set(square_pyramidal_numbers(limit)),
                "#00cec9"
            ),
            (
                "Pentagonal Numbers",
                set(pentagonal_numbers(limit)),
                "#00d2d3"
            ),
            (
                "Hexagonal Numbers",
                set(hexagonal_numbers(limit)),
                "#00a8ff"
            ),
            (
                "Perfect Numbers",
                {
                    n
                    for n in numbers
                    if n > 1 and proper_divisor_sum(n) == n
                },
                "#64d2ff"
            ),
            (
                "Abundant Numbers",
                {
                    n
                    for n in numbers
                    if proper_divisor_sum(n) > n
                },
                "#48dbfb"
            ),
            (
                "Deficient Numbers",
                {
                    n
                    for n in numbers
                    if proper_divisor_sum(n) < n
                },
                "#5ac8fa"
            ),
            (
                "Fibonacci Numbers",
                set(fibonacci_numbers(limit)),
                "#0984e3"
            ),
            (
                "Decimal Harshad Numbers",
                {n for n in numbers if is_harshad(n)},
                "#54a0ff"
            ),
            (
                "Sphenic Numbers",
                {n for n in numbers if is_sphenic(n)},
                "#2e86de"
            ),
            (
                "Smith Numbers",
                {n for n in numbers if is_smith(n)},
                "#686de0"
            ),
            (
                "Binary Palindromic Numbers",
                {
                    n
                    for n in numbers
                    if is_binary_palindrome(n)
                },
                "#0a84ff"
            ),
            (
                "Decimal Palindromic Numbers",
                {
                    n
                    for n in numbers
                    if is_decimal_palindrome(n)
                },
                "#6c5ce7"
            ),
            (
                "Happy Numbers",
                {n for n in numbers if is_happy(n)},
                "#bf5af2"
            ),
            (
                "Lucky Numbers",
                lucky_set,
                "#9b59b6"
            ),
            (
                "Evil Numbers",
                {n for n in numbers if is_evil(n)},
                "#ff2d55"
            ),
            (
                "Automorphic Numbers",
                {
                    n
                    for n in numbers
                    if is_automorphic(n)
                },
                "#e84393"
            ),
            (
                "Kaprekar Numbers",
                {n for n in numbers if is_kaprekar(n)},
                "#ff375f"
            ),
            (
                "Catalan Numbers",
                set(catalan_numbers(limit)),
                "#e17055"
            ),
            (
                "Pell Numbers",
                set(pell_numbers(limit)),
                "#ff453a"
            )
        ]

    # =====================================================
    # PREPARAÇÃO DOS ARQUIVOS
    # =====================================================
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    mp4_path = output_path / mp4_name
    gif_path = output_path / gif_name
    preview_path = output_path / preview_name

    numbers = list(range(1, 101))
    categories = build_categories(limit=100)

    frames_per_scene = reveal_frames + hold_frames
    total_frames = frames_per_scene * len(categories)

    # =====================================================
    # CONSTRUÇÃO DA FIGURA
    # =====================================================
    figure_width = width_pixels / dpi
    figure_height = height_pixels / dpi

    fig, ax = plt.subplots(
        figsize=(figure_width, figure_height),
        dpi=dpi
    )

    fig.patch.set_facecolor("black")
    ax.set_facecolor("black")

    ax.set_xlim(-0.6, 9.6)
    ax.set_ylim(-0.7, 11.9)
    ax.axis("off")

    title = ax.text(
        4.5,
        11.1,
        "",
        ha="center",
        va="center",
        fontsize=16,
        fontweight="bold",
        color="white"
    )

    subtitle = ax.text(
        4.5,
        10.55,
        "",
        ha="center",
        va="center",
        fontsize=8.5,
        color="#bcbcbc"
    )

    footer = ax.text(
        4.5,
        -0.35,
        "Numbers 1 to 100",
        ha="center",
        va="center",
        fontsize=8.5,
        color="#888888"
    )

    text_objects = {}

    for number in numbers:
        row = (number - 1) // 10
        column = (number - 1) % 10

        x_position = column
        y_position = 9 - row

        text_objects[number] = ax.text(
            x_position,
            y_position,
            str(number),
            ha="center",
            va="center",
            fontsize=9.8,
            fontweight="bold",
            color="#666666",
            bbox={
                "boxstyle": "round,pad=0.18",
                "facecolor": "#0d0d0d",
                "edgecolor": "#232323",
                "linewidth": 0.8
            }
        )

    # =====================================================
    # FUNÇÕES INTERNAS DA ANIMAÇÃO
    # =====================================================
    def initialize_animation():
        return [
            title,
            subtitle,
            footer,
            *text_objects.values()
        ]

    def update_animation(frame):
        scene_index = min(
            frame // frames_per_scene,
            len(categories) - 1
        )

        local_frame = frame % frames_per_scene

        scene_title, scene_values, scene_color = categories[
            scene_index
        ]

        ordered_values = sorted(scene_values)

        if local_frame < reveal_frames:
            number_of_active_values = max(
                1,
                math.ceil(
                    len(ordered_values)
                    * (local_frame + 1)
                    / reveal_frames
                )
            )

            active_values = set(
                ordered_values[:number_of_active_values]
            )
        else:
            active_values = scene_values

        title.set_text(scene_title)
        title.set_color(scene_color)

        subtitle.set_text(
            f"{len(scene_values)} highlighted"
        )

        for number, text_object in text_objects.items():
            box = text_object.get_bbox_patch()

            if number in active_values:
                text_object.set_color("white")

                box.set_facecolor(
                    to_rgba(scene_color, 0.28)
                )

                box.set_edgecolor(scene_color)
                box.set_linewidth(1.1)

            else:
                text_object.set_color("#6a6a6a")
                box.set_facecolor("#0d0d0d")
                box.set_edgecolor("#232323")
                box.set_linewidth(0.8)

        return [
            title,
            subtitle,
            footer,
            *text_objects.values()
        ]

    # =====================================================
    # CRIAÇÃO DA ANIMAÇÃO
    # =====================================================
    animation = FuncAnimation(
        fig,
        update_animation,
        init_func=initialize_animation,
        frames=total_frames,
        interval=1000 / fps_mp4,
        blit=False
    )

    generated_files = {}
    errors = {}

    # =====================================================
    # EXPORTAÇÃO DO PREVIEW
    # =====================================================
    if save_preview:
        try:
            update_animation(0)

            fig.savefig(
                preview_path,
                facecolor=fig.get_facecolor(),
                bbox_inches="tight",
                dpi=dpi
            )

            generated_files["preview"] = str(
                preview_path.resolve()
            )

        except Exception as error:
            errors["preview"] = str(error)

    # =====================================================
    # EXPORTAÇÃO DO MP4
    # =====================================================
    if save_mp4:
        try:
            mp4_writer = FFMpegWriter(
                fps=fps_mp4,
                bitrate=1800
            )

            animation.save(
                mp4_path,
                writer=mp4_writer,
                dpi=dpi
            )

            generated_files["mp4"] = str(
                mp4_path.resolve()
            )

        except Exception as error:
            errors["mp4"] = str(error)

    # =====================================================
    # EXPORTAÇÃO DO GIF
    # =====================================================
    if save_gif:
        try:
            gif_writer = PillowWriter(
                fps=fps_gif
            )

            animation.save(
                gif_path,
                writer=gif_writer,
                dpi=dpi
            )

            generated_files["gif"] = str(
                gif_path.resolve()
            )

        except Exception as error:
            errors["gif"] = str(error)

    plt.close(fig)

    # =====================================================
    # IMPRESSÃO OPCIONAL DAS CATEGORIAS
    # =====================================================
    if print_categories:
        for category_name, category_values, _ in categories:
            print(
                f"{category_name}: "
                f"{sorted(category_values)}"
            )

    return {
        "generated_files": generated_files,
        "errors": errors,
        "number_of_categories": len(categories),
        "number_of_frames": total_frames,
        "resolution": (
            width_pixels,
            height_pixels
        )
    }


def handler():
    result = generate_number_animation(
        output_dir=".",
        mp4_name="numbers_categories_9x16.mp4",
        gif_name="numbers_categories_9x16.gif",
        preview_name="numbers_categories_preview.png",
        width_pixels=360,
        height_pixels=640,
        dpi=100,
        fps_mp4=10,
        fps_gif=8,
        reveal_frames=3,
        hold_frames=4,
        save_mp4=True,
        save_gif=True,
        save_preview=True,
        print_categories=True
    )

    print("\nResultado da execução:")

    for file_type, file_path in result["generated_files"].items():
        print(f"{file_type.upper()}: {file_path}")

    for file_type, error in result["errors"].items():
        print(f"Erro na exportação {file_type.upper()}: {error}")

    print(
        f"Categorias: {result['number_of_categories']}"
    )

    print(
        f"Quadros: {result['number_of_frames']}"
    )

    print(
        f"Resolução: "
        f"{result['resolution'][0]}x"
        f"{result['resolution'][1]}"
    )

    return result

In [2]:
result = generate_number_animation(
    save_mp4=True,
    save_gif=False,
    save_preview=True
)